In [ ]:
# YOLOv8n Surgical Instrument Detection

Reproducible experimental pipeline for surgical instrument detection using YOLOv8n, including dataset preparation, training, independent test evaluation, computational benchmarking, mobile export, and INT8 quantization.

**Classes:** Scalpel, Forceps, Straight Scissors, Curved Scissors.

**Dataset:** Roboflow Universe — Surgical tools 2, Version 1.  
https://universe.roboflow.com/school-ratfh/surgical-tools-2/dataset/1

## 1. Environment and paths

In [1]:

!pip install -q ultralytics

import os
import re
import random
import shutil
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import psutil
import torch
import yaml

from ultralytics import YOLO

SEED = 42
random.seed(SEED)

# Dataset and output paths
SOURCE = Path("/kaggle/input/datasets/ainhoa0312/surgical-data")
OUTPUT = Path("/kaggle/working/surgical_data_70_10_21")
PROJECT_DIR = Path("/kaggle/working/eera_yolo_results")

CLASS_NAMES = {
    0: "Scalpel",
    1: "Forceps",
    2: "Straight Scissors",
    3: "Curved Scissors",
}

print("Dataset source:", SOURCE)
print("Source exists:", SOURCE.exists())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 648.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Dataset source: /kaggle/input/datasets/ainhoa0312/surgical-data
Source exists: True


## 2. Dataset inspection

In [3]:
def count_instances(labels_dir):
    counts = Counter()

    for label_file in Path(labels_dir).glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                if line.strip():
                    counts[int(line.split()[0])] += 1

    return counts


for split in ["train", "valid", "test"]:

    images_dir = SOURCE / split / "images"
    labels_dir = SOURCE / split / "labels"

    print(f"\n{split.upper()}")
    print("Images:", len(list(images_dir.glob("*"))))
    print("Instances:", count_instances(labels_dir))


TRAIN
Images: 5712
Instances: Counter({3: 2139, 1: 2004, 0: 1962, 2: 1905})

VALID
Images: 728
Instances: Counter({3: 269, 1: 255, 2: 253, 0: 249})

TEST
Images: 82
Instances: Counter({2: 37, 1: 33, 3: 26, 0: 24})


## 3. Data leakage analysis

In [4]:
def original_id(filename):

    name = os.path.basename(filename)
    name = os.path.splitext(name)[0]

    name = re.sub(r"\.rf\.[a-f0-9]+$", "", name)
    name = re.sub(
        r"_(jpg|jpeg|png)$",
        "",
        name,
        flags=re.IGNORECASE
    )

    return name


split_groups = {}

for split in ["train", "valid", "test"]:

    files = [
        f.name
        for f in (SOURCE / split / "images").iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    split_groups[split] = {
        original_id(f) for f in files
    }

    print(
        f"{split.upper()}: "
        f"{len(files)} images / "
        f"{len(split_groups[split])} unique originals"
    )


print(
    "\nOriginals shared TRAIN-VALID:",
    len(split_groups["train"] & split_groups["valid"])
)

print(
    "Originals shared TRAIN-TEST:",
    len(split_groups["train"] & split_groups["test"])
)

print(
    "Originals shared VALID-TEST:",
    len(split_groups["valid"] & split_groups["test"])
)

TRAIN: 5712 images / 1831 unique originals
VALID: 728 images / 705 unique originals
TEST: 82 images / 79 unique originals

Originals shared TRAIN-VALID: 0
Originals shared TRAIN-TEST: 0
Originals shared VALID-TEST: 0


## 4. Dataset reorganization (70/10/20)

In [5]:
groups = defaultdict(list)

for old_split in ["train", "valid", "test"]:

    images_dir = SOURCE / old_split / "images"
    labels_dir = SOURCE / old_split / "labels"

    for image_file in images_dir.iterdir():

        if image_file.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        oid = original_id(image_file.name)

        groups[oid].append({
            "image": image_file,
            "label": labels_dir / f"{image_file.stem}.txt"
        })


class_groups = defaultdict(list)

for oid in groups:

    name = oid.lower()

    if name.startswith("bisturi"):
        cls = 0

    elif name.startswith("pinca"):
        cls = 1

    elif name.startswith("tesourareta"):
        cls = 2

    elif name.startswith("tesouracurva"):
        cls = 3

    else:
        cls = -1

    class_groups[cls].append(oid)


for cls, ids in sorted(class_groups.items()):

    label = CLASS_NAMES.get(cls, "Mixed/other")

    print(
        f"{label}: {len(ids)} unique originals"
    )

Mixed/other: 99 unique originals
Scalpel: 630 unique originals
Forceps: 629 unique originals
Straight Scissors: 627 unique originals
Curved Scissors: 630 unique originals


In [ ]:
train_ids = []
val_ids = []
test_ids = []

random.seed(SEED)

for cls in [0, 1, 2, 3, -1]:

    ids = class_groups[cls].copy()
    random.shuffle(ids)

    n = len(ids)

    n_train = int(n * 0.70)
    n_val = int(n * 0.10)

    train_ids.extend(
        ids[:n_train]
    )

    val_ids.extend(
        ids[n_train:n_train + n_val]
    )

    test_ids.extend(
        ids[n_train + n_val:]
    )


print("\nUnique originals")

print("Train:", len(train_ids))
print("Valid:", len(val_ids))
print("Test:", len(test_ids))

print(
    "Train ∩ Valid:",
    len(set(train_ids) & set(val_ids))
)

print(
    "Train ∩ Test:",
    len(set(train_ids) & set(test_ids))
)

print(
    "Valid ∩ Test:",
    len(set(val_ids) & set(test_ids))
)

In [ ]:
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)


for split in ["train", "valid", "test"]:

    (OUTPUT / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


def copy_groups(ids, destination_split):

    image_dest = OUTPUT / destination_split / "images"
    label_dest = OUTPUT / destination_split / "labels"

    copied_images = 0
    copied_labels = 0

    for oid in ids:

        for item in groups[oid]:

            shutil.copy2(
                item["image"],
                image_dest / item["image"].name
            )

            copied_images += 1

            if item["label"].exists():

                shutil.copy2(
                    item["label"],
                    label_dest / item["label"].name
                )

                copied_labels += 1

    return copied_images, copied_labels


print(
    "TRAIN:",
    copy_groups(train_ids, "train")
)

print(
    "VALID:",
    copy_groups(val_ids, "valid")
)

print(
    "TEST:",
    copy_groups(test_ids, "test")
)

In [ ]:
for split in ["train", "valid", "test"]:

    counts = count_instances(
        OUTPUT / split / "labels"
    )

    total = sum(counts.values())

    print(f"\n===== {split.upper()} =====")

    print(
        "Images:",
        len(
            list(
                (OUTPUT / split / "images").glob("*")
            )
        )
    )

    for cls in range(4):

        n = counts[cls]

        pct = (
            100 * n / total
            if total else 0
        )

        print(
            f"{CLASS_NAMES[cls]:20s}: "
            f"{n:5d} ({pct:.1f}%)"
        )

In [ ]:
data_yaml = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": [
        CLASS_NAMES[i]
        for i in range(4)
    ],
}


with open(
    OUTPUT / "data.yaml",
    "w"
) as f:

    yaml.dump(
        data_yaml,
        f,
        sort_keys=False,
        default_flow_style=False
    )


print(
    open(
        OUTPUT / "data.yaml"
    ).read()
)

## 5. YOLOv8n training

In [ ]:
model = YOLO("yolov8n.pt")


train_results = model.train(

    data=str(
        OUTPUT / "data.yaml"
    ),

    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,

    seed=SEED,
    device=0,

    degrees=10,
    translate=0.05,
    scale=0.10,
    fliplr=0.5,

    mosaic=0.3,
    mixup=0.0,

    project=str(PROJECT_DIR),
    name="yolov8n_640",

    exist_ok=True,
)


BEST_PT = (
    PROJECT_DIR
    / "yolov8n_640"
    / "weights"
    / "best.pt"
)

print(
    "Best model:",
    BEST_PT
)


## 6. Independent test-set evaluation

In [ ]:

best_model = YOLO(
    str(BEST_PT)
)


test_results = best_model.val(

    data=str(
        OUTPUT / "data.yaml"
    ),

    split="test",

    imgsz=640,
    batch=16,

    conf=0.25,
    iou=0.6,

    plots=True,

    project=str(PROJECT_DIR),
    name="yolov8n_test",

    exist_ok=True,
)


print("\n===== YOLOv8n TEST =====")

print(
    f"Precision:     "
    f"{test_results.box.mp:.4f}"
)

print(
    f"Recall:        "
    f"{test_results.box.mr:.4f}"
)

print(
    f"mAP@0.5:      "
    f"{test_results.box.map50:.4f}"
)

print(
    f"mAP@0.5:0.95: "
    f"{test_results.box.map:.4f}"
)

## 7. Normalized confusion matrix

In [ ]:

cm = test_results.confusion_matrix.matrix.astype(float)

classes = [
    "Scalpel",
    "Forceps",
    "Straight Scissors",
    "Curved Scissors",
    "Background"
]


column_sums = cm.sum(
    axis=0,
    keepdims=True
)

cm_normalized = np.divide(
    cm,
    column_sums,
    out=np.zeros_like(cm),
    where=column_sums != 0
)


fig, ax = plt.subplots(
    figsize=(10, 8)
)

im = ax.imshow(
    cm_normalized,
    cmap="Blues",
    vmin=0,
    vmax=1
)


ax.set_xticks(
    np.arange(len(classes))
)

ax.set_yticks(
    np.arange(len(classes))
)

ax.set_xticklabels(
    classes,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(classes)

ax.set_xlabel("True Class")
ax.set_ylabel("Predicted Class")

ax.set_title(
    "Normalized Confusion Matrix – YOLOv8n"
)


for i in range(
    cm_normalized.shape[0]
):

    for j in range(
        cm_normalized.shape[1]
    ):

        value = cm_normalized[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color=(
                "white"
                if value > 0.5
                else "black"
            )
        )


cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Normalized Value"
)

plt.tight_layout()


CONFUSION_PATH = Path(
    "/kaggle/working/"
    "confusion_matrix_YOLOv8n_paper.png"
)

plt.savefig(
    CONFUSION_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved to:",
    CONFUSION_PATH
)

## 8. Python computational benchmark

In [ ]:
benchmark_model = YOLO(
    str(BEST_PT)
)

process = psutil.Process()

ram_before = (
    process.memory_info().rss
    / (1024 ** 2)
)


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


start = time.time()


predictions = benchmark_model.predict(

    source=str(
        OUTPUT / "test" / "images"
    ),

    imgsz=640,
    device=0,
    verbose=False,
)


end = time.time()


ram_after = (
    process.memory_info().rss
    / (1024 ** 2)
)

vram_peak = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


total_time = end - start

n_images = len(predictions)

ms_per_image = (
    total_time / n_images
) * 1000

fps = (
    n_images / total_time
)


print(
    "===== PYTHON PERFORMANCE ====="
)

print(
    f"Images: {n_images}"
)

print(
    f"Total time: "
    f"{total_time:.2f} s"
)

print(
    f"Average inference: "
    f"{ms_per_image:.2f} ms/image"
)

print(
    f"FPS: {fps:.2f}"
)

print(
    f"RAM before: "
    f"{ram_before:.2f} MB"
)

print(
    f"RAM after: "
    f"{ram_after:.2f} MB"
)

print(
    f"RAM increase: "
    f"{ram_after - ram_before:.2f} MB"
)

print(
    f"Peak VRAM: "
    f"{vram_peak:.2f} MB"
)

## 9. Mobile model export (FP32 LiteRT/TFLite)

In [ ]:
fp32_export = best_model.export(

    format="tflite",
    imgsz=640

)


print(
    "FP32 mobile model:",
    fp32_export
)

print(
    f"Size: "
    f"{os.path.getsize(fp32_export) / (1024 ** 2):.2f} MB"
)

## 10. INT8 quantization

The trained model is exported using INT8 quantization for resource-constrained
mobile deployment. Calibration is performed using the dataset configuration
and the same 640 × 640 input resolution.

In [ ]:
WORKING_PT = Path(
    "/kaggle/working/"
    "YOLOv8n_SurgicalTools_best.pt"
)


shutil.copy2(
    BEST_PT,
    WORKING_PT
)


quant_model = YOLO(
    str(WORKING_PT)
)


int8_export = quant_model.export(

    format="tflite",

    imgsz=640,

    quantize=8,

    data=str(
        OUTPUT / "data.yaml"
    ),

    fraction=0.25,
)


print(
    "INT8 mobile model:",
    int8_export
)

print(
    f"Size: "
    f"{os.path.getsize(int8_export) / (1024 ** 2):.2f} MB"
)

## 11. INT8 test-set evaluation

In [ ]:
int8_model = YOLO(
    str(int8_export)
)


int8_results = int8_model.val(

    data=str(
        OUTPUT / "data.yaml"
    ),

    split="test",

    imgsz=640,
    batch=1,

    conf=0.25,
    iou=0.6,

    plots=True,

    project=str(PROJECT_DIR),

    name="yolov8n_int8_test",

    exist_ok=True,
)


print(
    "\n===== INT8 TEST RESULTS ====="
)

print(
    f"Precision:     "
    f"{int8_results.box.mp:.3f}"
)

print(
    f"Recall:        "
    f"{int8_results.box.mr:.3f}"
)

print(
    f"mAP@0.5:      "
    f"{int8_results.box.map50:.3f}"
)

print(
    f"mAP@0.5:0.95: "
    f"{int8_results.box.map:.3f}"
)